# Notebook 11: Trained baselines (BPJFNN and a fine-tuned cross-encoder)

**Before running:** Runtime → Change runtime type → **T4 GPU**. Close any other GPU session first (Runtime → Manage sessions → Terminate).

**Run time:** about 2–3 hours (an estimate; nothing was timed in advance). **Resumable:** every finished run saves its scores to Drive (`results/11_cache/`). If Colab disconnects, just **Run all** again and it skips what is already done.

**Why this notebook:** the paper so far has one trained model (fine-tuned BGE). This notebook adds two more trained model families, so the benchmark findings do not rest on a single trained model:

| Model | What it is | Settings |
|---|---|---|
| **BPJFNN** | The basic person–job fit network of Qin et al. (SIGIR 2018): word embeddings → one BiLSTM for the job, one for the resume → mean pooling → a matching layer on [job; resume; job − resume] → softmax. **Our re-implementation from the paper's description; no official code is available.** | Trained from scratch on this dataset |
| **Cross-encoder** | `BAAI/bge-reranker-base` fine-tuned on the training pairs. It reads the job and the resume together (512 tokens shared). | Same recipe as the fine-tuned BGE bi-encoder: 2 epochs, lr 2e-5, batch 16, 10% warm-up, fp16, seed 42 |

Each model is trained and scored in exactly the two designs already used in the paper:

1. **Standard vs strict split** (Notebook 3): the 45 usable test jobs, NDCG@10 and the other ranking metrics.
2. **5 × 5 grid** (Notebook 6a): standard / size-matched / strict training sets on the same folds and the same test pairs as the fine-tuned BGE.

**All settings in this notebook were fixed before the first run** (see ANALYSIS-PLAN.md, addendum of 24 Nov 2026). The only value chosen from data is the number of BPJFNN epochs, chosen on the **validation** jobs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# --no-deps: do not let pip change the numpy version that is already loaded
!pip -q install --no-deps gensim smart_open

import os, re, gc, json, math, time, sys
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score

assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU, then run again'
device = 'cuda'
print(torch.cuda.get_device_name(0))

DATA = '/content/drive/MyDrive/resume-job-matching/data'
RESULTS = '/content/drive/MyDrive/resume-job-matching/results'
CACHE = f'{RESULTS}/11_cache'
os.makedirs(CACHE, exist_ok=True)
SEED = 42; K = 5

train_df = pd.read_parquet(f'{DATA}/train.parquet')
val_df = pd.read_parquet(f'{DATA}/val.parquet')
test_df = pd.read_parquet(f'{DATA}/test.parquet')
for d in (train_df, val_df, test_df):
    d['pair_key'] = d.job_id + '|' + d.resume_id
print(len(train_df), len(val_df), len(test_df))

# strict training set of Notebook 3: no test resume is seen in training
strict_train = train_df[~train_df.resume_id.isin(set(test_df.resume_id))]
SPLIT_TRAINS = {'standard': train_df, 'strict': strict_train}
print('strict training pairs:', len(strict_train))

# 5 x 5 grid, built exactly as in Notebook 6a
all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
rng = np.random.default_rng(SEED)
jobs = all_df.job_id.unique(); res = all_df.resume_id.unique()
job_fold = dict(zip(jobs, rng.permutation(np.arange(len(jobs)) % K)))
res_fold = dict(zip(res, rng.permutation(np.arange(len(res)) % K)))
all_df['jf'] = all_df.job_id.map(job_fold); all_df['rf'] = all_df.resume_id.map(res_fold)

def fold_sets(k):
    test = all_df[(all_df.jf == k) & (all_df.rf == k)]
    standard = all_df[all_df.jf != k]
    strict = all_df[(all_df.jf != k) & (all_df.rf != k)]
    matched = standard.sample(n=len(strict), random_state=SEED + k)
    return test, {'standard': standard, 'strict': strict, 'size-matched': matched}

# CHECK: the grid test pairs must be the same pairs Notebook 6a scored
old_grid = pd.read_parquet(f'{RESULTS}/06a_cv_pair_scores.parquet')
grid_test = pd.concat([fold_sets(k)[0].assign(fold=k) for k in range(K)], ignore_index=True)
assert set(grid_test.pair_key) == set(old_grid.pair_key), 'Grid folds differ from Notebook 6a - stop'
assert grid_test.pair_key.is_unique
print(f'grid check passed: {len(grid_test)} test pairs, same as Notebook 6a')

## 1. Metrics (copied from Notebooks 3 and 6a, unchanged)

In [ ]:
def dcg(rels):
    rels = np.asarray(rels, dtype=float)
    return np.sum((2 ** rels - 1) / np.log2(np.arange(2, len(rels) + 2)))

def ndcg_at_k(ranked_rels, k):
    ideal = dcg(sorted(ranked_rels, reverse=True)[:k])
    return dcg(ranked_rels[:k]) / ideal if ideal > 0 else 0.0

def precision_at_k(ranked_rels, k, min_rel=2):
    return np.mean([r >= min_rel for r in ranked_rels[:k]])

def mrr(ranked_rels, min_rel=2):
    for i, r in enumerate(ranked_rels):
        if r >= min_rel:
            return 1.0 / (i + 1)
    return 0.0

def score_query(ranked_rels):
    return {'NDCG@5': ndcg_at_k(ranked_rels, 5), 'NDCG@10': ndcg_at_k(ranked_rels, 10),
            'P@5': precision_at_k(ranked_rels, 5), 'MRR': mrr(ranked_rels)}

def bootstrap_ci(values, n=2000, seed=SEED):
    r = np.random.default_rng(seed)
    values = np.asarray(values, dtype=float)
    means = [r.choice(values, len(values), replace=True).mean() for _ in range(n)]
    return np.percentile(means, [2.5, 97.5])

def evaluate_scores(method_name, df, scores, split_name='test'):
    """Same loop as Notebook 3's evaluate(), but the scores are already computed (one per row of df)."""
    d = df.assign(model_score=np.asarray(scores, dtype=float))
    r = np.random.default_rng(SEED); rows = []
    for job_id, group in d[d.eval_usable].groupby('job_id'):
        s = group['model_score'].to_numpy()
        order = np.lexsort((r.random(len(s)), -s))
        ranked_rels = group.relevance.to_numpy()[order].tolist()
        rows.append({'method': method_name, 'split': split_name, 'job_id': job_id,
                     'candidates': len(group), **score_query(ranked_rels)})
    return pd.DataFrame(rows)

def paired_permutation_p(a, b, n=10000, seed=SEED):
    r = np.random.default_rng(seed)
    diff = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    signs = r.choice([-1, 1], size=(n, len(diff)))
    return float((np.abs((signs * diff).mean(axis=1)) >= abs(diff.mean())).mean())

def holm(pvals):
    p = np.asarray(pvals, dtype=float); order = np.argsort(p); k = len(p)
    adj = np.minimum(1, np.maximum.accumulate([(k - i) * p[j] for i, j in enumerate(order)]))
    out = np.empty(k); out[order] = adj
    return out

# --- from Notebook 6a ---
def auc(y, s):
    y = np.asarray(y); return roc_auc_score(y, s) if 0 < y.sum() < len(y) else np.nan

def pooled(df, col):
    fit = df[df.relevance > 0]
    return {'AUC fit-vs-no': auc((df.relevance > 0).astype(int), df[col]),
            'AUC good-vs-potential': auc((fit.relevance == 2).astype(int), fit[col])}

def per_job(df, col):
    r = np.random.default_rng(SEED); rows = []
    for jid, g in df.groupby('job_id'):
        rel = g.relevance.to_numpy(); s = g[col].to_numpy(); fit = rel > 0
        row = {'job_id': jid}
        if len(g) >= 3 and len(set(rel)) >= 2:
            row['NDCG@5'] = ndcg_at_k(rel[np.lexsort((r.random(len(s)), -s))].tolist(), 5)
        row['job AUC fit-vs-no'] = auc(fit.astype(int), s)
        row['job AUC good-vs-potential'] = auc((rel[fit] == 2).astype(int), s[fit]) if fit.sum() >= 2 else np.nan
        rows.append(row)
    return pd.DataFrame(rows).set_index('job_id')

def free(m):
    del m; gc.collect(); torch.cuda.empty_cache()

run_log = []   # one row per training run: model, design, training pairs, minutes

## 2. BPJFNN (Qin et al., SIGIR 2018), our re-implementation

Built from the paper's description: both texts are read as one long word sequence each; word embeddings (100-d, skip-gram pre-trained on the training texts, then trained further); one BiLSTM per side (hidden size 200); mean pooling; a matching layer `tanh(W [job; resume; job − resume])`; softmax.

**Where we differ from the paper, all fixed in advance:**
- **Three labels**, not two: softmax over No / Potential / Good. Ranking score = 2·P(Good) + P(Potential), the same form as the LLM judge.
- **Text length:** jobs cut at 500 words, resumes at 1,000 words.
- **Vocabulary and skip-gram** come from each run's **training texts only** (words seen at least twice). So a strict run never sees a test resume, not even as unlabelled text.
- **Taken from the paper:** 100-d skip-gram embeddings trained on the study's own texts, BiLSTM hidden size 200, Adam, batch 64 (the paper gives these for its main model, APJFNN; we assume they also hold for BPJFNN).
- **Not stated or unclear in the paper, so chosen here:** learning rate 1e-3, dropout 0.2 (the paper says "probability 0.8" without saying whether that is kept or dropped), gradient clipping at 5, matching layer size 200.
- **Epochs:** one run on the standard training set (seed 42) for 20 epochs; the epoch with the best **validation** NDCG@10 is used for every other run.
- A from-scratch model is noisy, so the standard and strict runs are repeated with **5 seeds**.

If the `gensim` library does not load in this Colab, the embeddings start random instead of skip-gram. The cell prints which one was used, and it is saved with the results.

In [ ]:
try:
    from gensim.models import Word2Vec
    HAVE_GENSIM = True
except Exception as e:
    HAVE_GENSIM = False
    print('gensim not available:', type(e).__name__, e)
print('Embedding start:', 'skip-gram (gensim)' if HAVE_GENSIM else 'RANDOM (gensim did not load)')

MAX_JOB, MAX_RES = 500, 1000
EMB, HID, BP_BATCH, BP_LR, BP_DROPOUT, BP_MAX_EPOCHS, BP_CLIP = 100, 200, 64, 1e-3, 0.2, 20, 5.0
BP_SEEDS = [42, 43, 44, 45, 46]

TOKEN_RE = re.compile(r'[a-z0-9][a-z0-9+#.]*')   # same pattern as the BM25 tokeniser; stop words are kept
def tokenize(text):
    return TOKEN_RE.findall(text.lower())

job_tok = {j: tokenize(t)[:MAX_JOB] for j, t in all_df.drop_duplicates('job_id')[['job_id', 'job_text']].itertuples(index=False)}
res_tok = {r: tokenize(t)[:MAX_RES] for r, t in all_df.drop_duplicates('resume_id')[['resume_id', 'resume_text']].itertuples(index=False)}
print(len(job_tok), 'jobs,', len(res_tok), 'resumes tokenised')

def build_vocab(train):
    c = Counter()
    for j in train.job_id.unique(): c.update(job_tok[j])
    for r in train.resume_id.unique(): c.update(res_tok[r])
    words = sorted(w for w, n in c.items() if n >= 2)
    return {w: i + 2 for i, w in enumerate(words)}          # 0 = padding, 1 = unknown word

def start_embeddings(train, vocab, seed):
    g = torch.Generator().manual_seed(seed)
    emb = (torch.rand(len(vocab) + 2, EMB, generator=g) - 0.5) * 0.2
    emb[0] = 0
    if HAVE_GENSIM:
        sents = [job_tok[j] for j in sorted(train.job_id.unique())] + [res_tok[r] for r in sorted(train.resume_id.unique())]
        w2v = Word2Vec(sents, vector_size=EMB, sg=1, window=5, min_count=2, epochs=5, seed=seed, workers=1)
        for w, i in vocab.items():
            if w in w2v.wv:
                emb[i] = torch.tensor(np.array(w2v.wv[w]))
    return emb

class BPJFNN(nn.Module):
    def __init__(self, emb):
        super().__init__()
        self.emb = nn.Embedding.from_pretrained(emb.clone(), freeze=False, padding_idx=0)
        self.job_lstm = nn.LSTM(EMB, HID, batch_first=True, bidirectional=True)
        self.res_lstm = nn.LSTM(EMB, HID, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(BP_DROPOUT)
        self.match = nn.Linear(6 * HID, HID)
        self.out = nn.Linear(HID, 3)

    def encode(self, lstm, x):
        mask = x != 0
        lengths = mask.sum(1).clamp(min=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(self.drop(self.emb(x)), lengths, batch_first=True, enforce_sorted=False)
        h, _ = lstm(packed)
        h, _ = nn.utils.rnn.pad_packed_sequence(h, batch_first=True, total_length=x.size(1))
        m = mask.unsqueeze(-1).float()
        return (h * m).sum(1) / m.sum(1).clamp(min=1)       # mean over the real words only

    def forward(self, j, r):
        sj, sr = self.encode(self.job_lstm, j), self.encode(self.res_lstm, r)
        d = torch.tanh(self.match(self.drop(torch.cat([sj, sr, sj - sr], dim=1))))
        return self.out(d)

def pad(seqs):
    out = torch.zeros(len(seqs), max(len(s) for s in seqs), dtype=torch.long)
    for i, s in enumerate(seqs):
        out[i, :len(s)] = torch.tensor(s, dtype=torch.long)
    return out.to(device)

def bp_score(model, df, jid, rid, bs=128):
    model.eval(); out = []
    J = df.job_id.to_numpy(); R = df.resume_id.to_numpy()
    with torch.no_grad():
        for i in range(0, len(df), bs):
            p = torch.softmax(model(pad([jid[j] for j in J[i:i + bs]]), pad([rid[r] for r in R[i:i + bs]])), dim=1)
            out.append((2 * p[:, 2] + p[:, 1]).float().cpu().numpy())
    return np.concatenate(out)

def bp_run(train, eval_sets, seed, epochs, track_val=None):
    """Train one BPJFNN on `train`; return scores for each dataframe in eval_sets."""
    torch.manual_seed(seed); np.random.seed(seed)
    vocab = build_vocab(train)
    emb = start_embeddings(train, vocab, seed)
    jid = {j: ([vocab.get(w, 1) for w in t] or [1]) for j, t in job_tok.items()}
    rid = {r: ([vocab.get(w, 1) for w in t] or [1]) for r, t in res_tok.items()}
    model = BPJFNN(emb).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=BP_LR)
    lossf = nn.CrossEntropyLoss()
    J = train.job_id.to_numpy(); R = train.resume_id.to_numpy(); Y = train.relevance.to_numpy().astype('int64')
    r = np.random.default_rng(seed); history = []
    for ep in range(epochs):
        model.train()
        order = r.permutation(len(train))
        for i in range(0, len(order), BP_BATCH):
            b = order[i:i + BP_BATCH]
            logits = model(pad([jid[j] for j in J[b]]), pad([rid[x] for x in R[b]]))
            loss = lossf(logits, torch.tensor(Y[b], device=device))
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), BP_CLIP); opt.step()
        if track_val is not None:
            history.append(float(evaluate_scores('', track_val, bp_score(model, track_val, jid, rid), 'val')['NDCG@10'].mean()))
            print(f'  epoch {ep + 1:2d}  validation NDCG@10 {history[-1]:.4f}')
    outs = [bp_score(model, d, jid, rid) for d in eval_sets]
    free(model)
    return outs, history, len(vocab)

### 2a. Choose the number of epochs on the validation jobs

In [ ]:
ep_path = f'{CACHE}/bpjfnn_epochs.json'
if os.path.exists(ep_path):
    ep_info = json.load(open(ep_path))
    print('cached')
else:
    t0 = time.perf_counter()
    _, history, vocab_size = bp_run(train_df, [], SEED, BP_MAX_EPOCHS, track_val=val_df)
    ep_info = {'epochs': int(np.argmax(history)) + 1, 'val_ndcg10_by_epoch': history, 'vocab_size': vocab_size,
               'embedding_start': 'skip-gram' if HAVE_GENSIM else 'random',
               'minutes': round((time.perf_counter() - t0) / 60, 2)}
    json.dump(ep_info, open(ep_path, 'w'))
BP_EPOCHS = ep_info['epochs']
print(f"BPJFNN epochs = {BP_EPOCHS} (validation NDCG@10 {max(ep_info['val_ndcg10_by_epoch']):.4f}), "
      f"vocabulary {ep_info['vocab_size']} words, embedding start: {ep_info['embedding_start']}")
assert ep_info['embedding_start'] == ('skip-gram' if HAVE_GENSIM else 'random'), \
    'gensim availability changed between sessions - stop'

### 2b. Standard and strict split (5 seeds), then the 5 × 5 grid (seed 42)

In [ ]:
def cached_run(path, fn):
    """fn() -> dataframe. Runs only if the file is not on Drive yet."""
    if os.path.exists(path):
        return False
    t0 = time.perf_counter(); out = fn(); out.to_parquet(path, index=False)
    return round((time.perf_counter() - t0) / 60, 2)

def split_frame(scores_val, scores_test):
    return pd.concat([val_df[['pair_key']].assign(split='val', score=scores_val),
                      test_df[['pair_key']].assign(split='test', score=scores_test)], ignore_index=True)

for seed in BP_SEEDS:
    for name, train in SPLIT_TRAINS.items():
        path = f'{CACHE}/bpjfnn_split_{name}_s{seed}.parquet'
        mins = cached_run(path, lambda: split_frame(*bp_run(train, [val_df, test_df], seed, BP_EPOCHS)[0]))
        print(f'BPJFNN {name} seed {seed}:', 'cached' if mins is False else f'{len(train)} train pairs, {mins} min')
        if mins is not False:
            run_log.append({'model': 'BPJFNN', 'design': f'split {name} s{seed}', 'train pairs': len(train), 'minutes': mins})

for k in range(K):
    test, trains = fold_sets(k)
    for name, train in trains.items():
        path = f'{CACHE}/bpjfnn_fold{k}_{name}.parquet'
        mins = cached_run(path, lambda: test[['pair_key']].assign(score=bp_run(train, [test], SEED, BP_EPOCHS)[0][0]))
        print(f'BPJFNN fold {k} {name}:', 'cached' if mins is False else f'{len(train)} train pairs, {mins} min')
        if mins is not False:
            run_log.append({'model': 'BPJFNN', 'design': f'grid {k} {name}', 'train pairs': len(train), 'minutes': mins})

## 3. Fine-tuned cross-encoder (`BAAI/bge-reranker-base`)

The cross-encoder reads the job and the resume **together** in one input of at most 512 tokens. When both texts are long, the tokenizer cuts the longer one first, so each side keeps roughly half of the space.

Training target = relevance / 2 (0, 0.5, 1) with a binary cross-entropy loss. Everything else is the recipe of the fine-tuned BGE bi-encoder (Notebook 3): 2 epochs, learning rate 2e-5, batch 16, 10% warm-up, fp16, seed 42. **Fixed in advance, not tuned on test.**

This is the longest part of the notebook (17 training runs).

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

CE_BASE = 'BAAI/bge-reranker-base'
CE_MAXLEN, CE_EPOCHS, CE_LR, CE_BATCH = 512, 2, 2e-5, 16
ce_tok = AutoTokenizer.from_pretrained(CE_BASE)

def ce_encode(job_texts, resume_texts):
    return ce_tok(list(job_texts), list(resume_texts), truncation='longest_first', max_length=CE_MAXLEN,
                  padding=True, return_tensors='pt').to(device)

def ce_score(model, df, bs=64):
    model.eval()
    J = df.job_text.to_numpy(); R = df.resume_text.to_numpy()
    order = np.argsort([len(a) + len(b) for a, b in zip(J, R)])      # similar lengths together: faster
    out = np.zeros(len(df), dtype='float32')
    with torch.no_grad():
        for i in range(0, len(order), bs):
            b = order[i:i + bs]
            with torch.autocast('cuda', dtype=torch.float16):
                logits = model(**ce_encode(J[b], R[b])).logits.squeeze(-1)
            out[b] = logits.float().cpu().numpy()
    return out

def ce_run(train, eval_sets, seed=SEED):
    torch.manual_seed(seed); np.random.seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(CE_BASE, num_labels=1).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=CE_LR, weight_decay=0.0)
    steps = math.ceil(len(train) / CE_BATCH) * CE_EPOCHS
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    try:
        scaler = torch.amp.GradScaler('cuda')
    except Exception:
        scaler = torch.cuda.amp.GradScaler()          # older torch versions
    lossf = nn.BCEWithLogitsLoss()
    J = train.job_text.to_numpy(); R = train.resume_text.to_numpy()
    Y = (train.relevance / 2).to_numpy().astype('float32')
    r = np.random.default_rng(seed)
    for ep in range(CE_EPOCHS):
        model.train()
        order = r.permutation(len(train))
        for i in range(0, len(order), CE_BATCH):
            b = order[i:i + CE_BATCH]
            with torch.autocast('cuda', dtype=torch.float16):
                logits = model(**ce_encode(J[b], R[b])).logits.squeeze(-1)
            loss = lossf(logits.float(), torch.tensor(Y[b], device=device))
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
    outs = [ce_score(model, d) for d in eval_sets]
    free(model)
    return outs

for name, train in SPLIT_TRAINS.items():
    path = f'{CACHE}/ce_split_{name}.parquet'
    mins = cached_run(path, lambda: split_frame(*ce_run(train, [val_df, test_df])))
    print(f'Cross-encoder {name}:', 'cached' if mins is False else f'{len(train)} train pairs, {mins} min')
    if mins is not False:
        run_log.append({'model': 'Cross-encoder', 'design': f'split {name}', 'train pairs': len(train), 'minutes': mins})

for k in range(K):
    test, trains = fold_sets(k)
    for name, train in trains.items():
        path = f'{CACHE}/ce_fold{k}_{name}.parquet'
        mins = cached_run(path, lambda: test[['pair_key']].assign(score=ce_run(train, [test])[0]))
        print(f'Cross-encoder fold {k} {name}:', 'cached' if mins is False else f'{len(train)} train pairs, {mins} min')
        if mins is not False:
            run_log.append({'model': 'Cross-encoder', 'design': f'grid {k} {name}', 'train pairs': len(train), 'minutes': mins})

## 4. Result A: the 45 test jobs (standard and strict training)

The BPJFNN row is the **mean over its 5 seeds** (per job, then over jobs); the "seed SD" column shows how much the single-seed NDCG@10 moves. Rows marked *(Notebook 2/3)* are the numbers already in the paper, read from the saved files.

In [ ]:
def split_scores(path, split):
    d = pd.read_parquet(path); d = d[d.split == split]
    df = val_df if split == 'val' else test_df
    assert d.pair_key.tolist() == df.pair_key.tolist(), f'{path}: rows are not in the data order - stop'
    return d.score.to_numpy()

METRICS = ['NDCG@5', 'NDCG@10', 'P@5', 'MRR']
pq_new, seed_sd = [], {}
for name in SPLIT_TRAINS:
    label = 'BPJFNN' + (', strict' if name == 'strict' else '')
    runs = [evaluate_scores(label, test_df, split_scores(f'{CACHE}/bpjfnn_split_{name}_s{s}.parquet', 'test')) for s in BP_SEEDS]
    seed_sd[label] = float(np.std([r['NDCG@10'].mean() for r in runs], ddof=1))
    mean_run = runs[0].copy()
    for m in METRICS:
        mean_run[m] = np.mean([r[m].to_numpy() for r in runs], axis=0)     # same job order in every run
    pq_new.append(mean_run)
    label = 'Cross-encoder (fine-tuned' + (', strict)' if name == 'strict' else ')')
    pq_new.append(evaluate_scores(label, test_df, split_scores(f'{CACHE}/ce_split_{name}.parquet', 'test')))
pq_new = pd.concat(pq_new, ignore_index=True)

old_pq = pd.concat([pd.read_csv(f'{RESULTS}/02_baselines_per_query.csv'), pd.read_csv(f'{RESULTS}/03_embeddings_per_query.csv')])
OLD = ['TF-IDF', 'BM25', 'BGE-base (zero-shot)', 'CrossEnc-MiniLM (zero-shot)', 'BGE-base (fine-tuned)', 'BGE-base (fine-tuned, strict)']
old_pq = old_pq[(old_pq.split == 'test') & old_pq.method.isin(OLD)]
pq_all = pd.concat([old_pq, pq_new], ignore_index=True)

rows = []
for m, g in pq_all.groupby('method', sort=False):
    lo, hi = bootstrap_ci(g['NDCG@10'])
    rows.append({'method': m + ('' if m in set(pq_new.method) else '  (Notebook 2/3)'), 'jobs': len(g),
                 'NDCG@10': round(g['NDCG@10'].mean(), 4), 'NDCG@10 CI': f'[{lo:.3f}, {hi:.3f}]',
                 'NDCG@5': round(g['NDCG@5'].mean(), 4), 'P@5': round(g['P@5'].mean(), 3), 'MRR': round(g['MRR'].mean(), 3),
                 'seed SD (NDCG@10)': round(seed_sd[m], 4) if m in seed_sd else ''})
test_table = pd.DataFrame(rows).sort_values('NDCG@10', ascending=False).reset_index(drop=True)
display(test_table)

wide = pq_all.pivot(index='job_id', columns='method', values='NDCG@10')
assert wide.notna().all().all(), 'Some method is missing a test job - stop'
new_methods = list(pq_new.method.unique())
tests = [{'comparison': f'{m}  -  TF-IDF', 'diff': round((wide[m] - wide['TF-IDF']).mean(), 4),
          'p': paired_permutation_p(wide[m], wide['TF-IDF'])} for m in new_methods]
split_tests = pd.DataFrame(tests); split_tests['p (Holm, 4 tests)'] = holm(split_tests.p)
extra = [{'comparison': f'{a}  -  {b}', 'diff': round((wide[a] - wide[b]).mean(), 4), 'p': paired_permutation_p(wide[a], wide[b])}
         for a, b in [('BPJFNN', 'BPJFNN, strict'), ('Cross-encoder (fine-tuned)', 'Cross-encoder (fine-tuned, strict)'),
                      ('Cross-encoder (fine-tuned)', 'BGE-base (fine-tuned)'),
                      ('Cross-encoder (fine-tuned, strict)', 'BGE-base (fine-tuned, strict)')]]
split_tests = pd.concat([split_tests, pd.DataFrame(extra)], ignore_index=True)
print('Paired permutation tests on NDCG@10 over the 45 test jobs. Holm correction covers the four "- TF-IDF" tests only.')
display(split_tests)

val_check = {('BPJFNN s42' if 'bpjfnn' in p else 'Cross-encoder'): round(float(evaluate_scores('', val_df, split_scores(f'{CACHE}/{p}', 'val'), 'val')['NDCG@10'].mean()), 4)
             for p in ['bpjfnn_split_standard_s42.parquet', 'ce_split_standard.parquet']}
print('Validation NDCG@10 (standard training), for information:', val_check)

## 5. Result B: the 5 × 5 grid (does the memorisation finding hold for other trained models?)

Same test pairs as Notebook 6a. The main measure is the **per-job AUC, fit vs no fit** (jobs whose test list has both a fit and a no-fit candidate), as in the paper's memorisation table. The "FT" rows are the fine-tuned BGE numbers from Notebook 6a, recomputed here as a check: they must match the paper (0.840 / 0.823 / 0.736).

In [ ]:
ev = grid_test[['pair_key', 'job_id', 'resume_id', 'fold', 'relevance', 'label']].merge(
    old_grid[['pair_key', 'FT standard', 'FT size-matched', 'FT strict', 'TF-IDF']], on='pair_key').reset_index(drop=True)
FAMILIES = {'FT': 'Fine-tuned BGE (Notebook 6a)', 'BPJFNN': 'BPJFNN', 'CE': 'Cross-encoder'}
for fam, prefix in [('BPJFNN', 'bpjfnn'), ('CE', 'ce')]:
    for name in ['standard', 'size-matched', 'strict']:
        s = pd.concat([pd.read_parquet(f'{CACHE}/{prefix}_fold{k}_{name}.parquet') for k in range(K)])
        ev = ev.merge(s.rename(columns={'score': f'{fam} {name}'}), on='pair_key').reset_index(drop=True)
assert len(ev) == len(grid_test)

GRID_METHODS = [f'{fam} {n}' for fam in FAMILIES for n in ['standard', 'size-matched', 'strict']] + ['TF-IDF']
pj = {m: per_job(ev, m) for m in GRID_METHODS}
rows = []
for m in GRID_METHODS:
    a = pj[m]['job AUC fit-vs-no'].dropna(); g = pj[m]['job AUC good-vs-potential'].dropna()
    lo, hi = bootstrap_ci(a); glo, ghi = bootstrap_ci(g); p = pooled(ev, m)
    rows.append({'method': m, 'job AUC fit/no': round(a.mean(), 3), 'CI': f'[{lo:.3f}, {hi:.3f}]', 'jobs': len(a),
                 'pooled AUC fit/no': round(p['AUC fit-vs-no'], 3),
                 'job AUC good/potential': round(g.mean(), 3), 'G/P CI': f'[{glo:.3f}, {ghi:.3f}]', 'G/P jobs': len(g),
                 'NDCG@5': round(pj[m]['NDCG@5'].mean(), 3)})
grid_table = pd.DataFrame(rows)
print(f'{len(ev)} test pairs, {ev.job_id.nunique()} jobs over {K} folds')
display(grid_table)

check = grid_table.set_index('method').loc[['FT standard', 'FT size-matched', 'FT strict'], 'job AUC fit/no'].tolist()
print('CHECK against the paper (0.840, 0.823, 0.736):', check, '-> OK' if check == [0.840, 0.823, 0.736] else '-> DIFFERENT')

grid_tests = []
for fam in FAMILIES:
    fam_rows = []
    for a, b in [('standard', 'strict'), ('size-matched', 'strict'), ('standard', 'size-matched'), ('strict', 'TF-IDF')]:
        ca = f'{fam} {a}'; cb = 'TF-IDF' if b == 'TF-IDF' else f'{fam} {b}'
        both = pd.concat([pj[ca]['job AUC fit-vs-no'], pj[cb]['job AUC fit-vs-no']], axis=1).dropna()
        diff = both.iloc[:, 0] - both.iloc[:, 1]; lo, hi = bootstrap_ci(diff)
        fam_rows.append({'model': FAMILIES[fam], 'comparison': f'{a}  -  {b}', 'jobs': len(both), 'diff': round(diff.mean(), 3),
                         'CI': f'[{lo:.3f}, {hi:.3f}]', 'p': paired_permutation_p(both.iloc[:, 0], both.iloc[:, 1])})
    fam_rows = pd.DataFrame(fam_rows); fam_rows['p (Holm, 4 tests)'] = holm(fam_rows.p)
    grid_tests.append(fam_rows)
grid_tests = pd.concat(grid_tests, ignore_index=True)
print('Per-job AUC fit vs no fit: paired differences, sign-flip test, Holm within each model (4 tests).')
display(grid_tests)

### How to read this (the rules were fixed before the run)
- **size-matched − strict > 0 with Holm p < 0.05** for a new model → the memorisation finding holds for that model too. Otherwise the paper says it was not shown for that model.
- **job AUC good/potential, strict row:** if its CI includes 0.5, the new model also cannot tell Good from Potential on unseen resumes, as the paper says of the other methods.
- **Result A, "− TF-IDF" tests:** if a new model under **strict** training beats TF-IDF with Holm p < 0.05, the paper's recommendation about trained models on the 45 test jobs is reworded.
- Whatever the numbers are, both models are reported.

## 6. Save

In [ ]:
import transformers
test_table.to_csv(f'{RESULTS}/11_test_split_summary.csv', index=False)
pq_new.to_csv(f'{RESULTS}/11_test_split_per_query.csv', index=False)
split_tests.to_csv(f'{RESULTS}/11_test_split_tests.csv', index=False)
grid_table.to_csv(f'{RESULTS}/11_grid_summary.csv', index=False)
grid_tests.to_csv(f'{RESULTS}/11_grid_tests.csv', index=False)
ev.to_parquet(f'{RESULTS}/11_grid_pair_scores.parquet', index=False)

log_path = f'{RESULTS}/11_run_log.csv'
log = pd.DataFrame(run_log)
if os.path.exists(log_path):
    log = pd.concat([pd.read_csv(log_path), log], ignore_index=True)     # keep the timings of earlier sessions
log.to_csv(log_path, index=False)

info = {'bpjfnn': {**ep_info, 'max_job_words': MAX_JOB, 'max_resume_words': MAX_RES, 'emb': EMB, 'hidden': HID,
                   'batch': BP_BATCH, 'lr': BP_LR, 'dropout': BP_DROPOUT, 'clip': BP_CLIP, 'seeds': BP_SEEDS,
                   'seed_sd_ndcg10': seed_sd},
        'cross_encoder': {'base': CE_BASE, 'max_length': CE_MAXLEN, 'epochs': CE_EPOCHS, 'lr': CE_LR, 'batch': CE_BATCH},
        'validation_ndcg10': val_check,
        'versions': {'python': sys.version.split()[0], 'torch': torch.__version__, 'transformers': transformers.__version__,
                     'pandas': pd.__version__, 'numpy': np.__version__, 'gpu': torch.cuda.get_device_name(0)}}
json.dump(info, open(f'{RESULTS}/11_run_info.json', 'w'), indent=1)
if len(log):
    print(log.groupby('model').minutes.agg(['count', 'sum', 'median']).round(1))
print('saved')

## Done
Saved to `MyDrive/resume-job-matching/results/`: `11_test_split_summary.csv`, `11_test_split_per_query.csv`, `11_test_split_tests.csv`, `11_grid_summary.csv`, `11_grid_tests.csv`, `11_grid_pair_scores.parquet`, `11_run_log.csv`, `11_run_info.json`.

**Next:** File → Download → Download .ipynb.